<a href="https://colab.research.google.com/github/Kamilr616/AI_sign_language_translator/blob/develop/notebooks/Custom_gesture_recognizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project: /mediapipe/_project.yaml
Book: /mediapipe/_book.yaml


# Hand gesture recognition model customization

This notebook shows the end-to-end process of customizing a gesture recognizer model for recognizing ASL alphabet signs.

## Prerequisites

Install or update required packages.

In [1]:
!pip install --upgrade pip
!pip install mediapipe-model-maker
!pip install kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 51.4 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of tf-keras to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 134.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.3/475.3 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.8/611.8 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

Import required libraries.

In [2]:
import os
import tensorflow as tf
import matplotlib.pyplot as plt

assert tf.__version__.startswith('2')

from mediapipe_model_maker import gesture_recognizer

/usr/local/lib/python3.11/dist-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


###Upload kaggle.json file

In [3]:
from google.colab import files
files.upload() #kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"kamilrataj","key":"d2db69ee3b6478401271b492e85cdd6a"}'}

### Download and process the dataset

This code fetches data from kaggle directly. Uploading kaggle.json file is required.

In [13]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

import kaggle
import shutil

kaggle_dataset_name = "grassknoted/asl-alphabet"

try:
  kaggle.api.dataset_download_files(kaggle_dataset_name, path='datasets/', unzip=True)
  dataset_train_path = "datasets/asl_alphabet_train/asl_alphabet_train"
  os.rename(f"{dataset_train_path}/nothing", f"{dataset_train_path}/none")
  shutil.rmtree(f"{dataset_train_path}/del", ignore_errors=True)
  shutil.rmtree(f"{dataset_train_path}/space", ignore_errors=True)
  shutil.rmtree(f"{dataset_train_path}/J", ignore_errors=True)
  shutil.rmtree(f"{dataset_train_path}/Z", ignore_errors=True)
  print(f"Successfully processed {kaggle_dataset_name} files!")
except Exception as e:
  print(f"Failed to load {kaggle_dataset_name}: {e}!")


Dataset URL: https://www.kaggle.com/datasets/grassknoted/asl-alphabet
Successfully downloaded grassknoted/asl-alphabet!
Failed to load grassknoted/asl-alphabet: [Errno 39] Directory not empty: 'datasets/asl_alphabet_train/asl_alphabet_train/nothing' -> 'datasets/asl_alphabet_train/asl_alphabet_train/none'!


### Create the Dataset


**Load the dataset**

* `shuffle`: A boolean controlling whether to shuffle the dataset. Defaults to true.
* `min_detection_confidence`: A float between 0 and 1 controlling the confidence threshold for hand detection.

In [5]:
data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_train_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams(shuffle=True, min_detection_confidence=0.55)
)

### Print Dataset labels.
One of them should be the `none` gesture.


In [6]:
print(f"Labels: {data.label_names}")
print(f"Number of classes: {data.num_classes}")
print(f"Size :{data.size}")

Labels: ['none', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']
Number of classes: 25
Size :54811


**Split the dataset**

80% for training, 18% for validation, and 2% for testing.

In [7]:
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.9)

## Hyperparameters


`ModelOptions` customizable parameters:
* `dropout_rate`: The fraction of the input units to drop. Used in dropout layer. Defaults to 0.05.
* `layer_widths`: A list of hidden layer widths for the gesture model. Each element in the list will create a new hidden layer with the specified width. The hidden layers are separated with BatchNorm, Dropout, and ReLU. Defaults to an empty list(no hidden layers).

`HParams` customizable parameters:
* `learning_rate`: The learning rate to use for gradient descent training. Defaults to 0.001.
* `batch_size`: Batch size for training. Defaults to 2.
* `epochs`: Number of training iterations over the dataset. Defaults to 10.
* `steps_per_epoch`: An optional integer that indicates the number of training steps per epoch. If not set, the training pipeline calculates the default steps per epoch as the training dataset size divided by batch size.
* `shuffle`: True if the dataset is shuffled before training. Defaults to False.
* `lr_decay`: Learning rate decay to use for gradient descent training. Defaults to 0.99.
* `gamma`: Gamma parameter for focal loss. Defaults to 2

Additional `HParams` parameter that does not affect model accuracy:
* `export_dir`: The location of the model checkpoint files and exported model files.

**Set manual Hyperparameters**

In [8]:
_dropout_rate = 0.075
_layer_width = [128, 64, 32]
_learning_rate = 0.001
_batch_size = 16
_epochs = 70
_steps_per_epoch = None
_shuffle = True
_lr_decay = 0.95
_gamma = 2

**Train the model**

Train the custom gesture recognizer by using the create method and passing in the training data, validation data, model options, and hyperparameters.

In [9]:
hparams = gesture_recognizer.HParams(learning_rate=_learning_rate, export_dir="exported_model", batch_size=_batch_size, epochs=_epochs, steps_per_epoch=_steps_per_epoch, shuffle=_shuffle, lr_decay=_lr_decay, gamma=_gamma)

model_options = gesture_recognizer.ModelOptions(dropout_rate=_dropout_rate, layer_widths=_layer_width)
options = gesture_recognizer.GestureRecognizerOptions(model_options=model_options, hparams=hparams)
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hand_embedding (InputLayer  [(None, 128)]             0         
 )                                                               
                                                                 
 batch_normalization (Batch  (None, 128)               512       
 Normalization)                                                  
                                                                 
 re_lu (ReLU)                (None, 128)               0         
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 custom_gesture_recognizer_  (None, 128)               16512     
 0 (Dense)                                                       
                                                             

**Evaluate the model performance**

After training the model, evaluate it on a test dataset and print the loss and accuracy metrics.

In [10]:
loss, accuracy = model.evaluate(data=test_data, batch_size=16)
print(f"Test loss: {loss}, Test accuracy: {accuracy}")

275/275 [==============================] - 23s 6ms/step - loss: 0.0228 - categorical_accuracy: 0.9818
Test loss: 0.022805865854024887, Test accuracy: 0.9817684888839722


**Print model summary**

In [11]:
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hand_embedding (InputLayer  [(None, 128)]             0         
 )                                                               
                                                                 
 batch_normalization (Batch  (None, 128)               512       
 Normalization)                                                  
                                                                 
 re_lu (ReLU)                (None, 128)               0         
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 custom_gesture_recognizer_  (None, 128)               16512     
 0 (Dense)                                                       
                                                             

**Export to Tensorflow Lite Model**

In [12]:
model.export_model()
files.download('exported_model/gesture_recognizer.task')


Using existing files at /tmp/model_maker/gesture_recognizer/palm_detection_full.tflite
Using existing files at /tmp/model_maker/gesture_recognizer/hand_landmark_full.tflite


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

###Export and download labels

In [14]:
model.export_labels('exported_labels', 'labels.txt')
files.download('exported_labels/labels.txt')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>